In [1]:
import spacy
from Bio import Entrez
import pandas as pd
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [9]:
#1. Download papers from PubMed Central

from Bio import Entrez

Entrez.email = "jawahir_noor@hotmail.com"

handle = Entrez.esearch(
    db="pmc",
    term="genomic variants AND rare mutation AND human",
    retmax=5
)

record = Entrez.read(handle)
ids = record["IdList"]

fetch = Entrez.efetch(
    db="pmc",
    id=ids,
    rettype="xml",
    retmode="text"
)

papers_xml = fetch.read()

#Parse each paper properly
from bs4 import BeautifulSoup

soup = BeautifulSoup(papers_xml, "lxml")

papers_list = []

articles = soup.find_all("article")

for article in articles:
    title = article.find("article-title")
    abstract = article.find("abstract")

    papers_list.append({
        "title": title.get_text(" ", strip=True) if title else "",
        "abstract": abstract.get_text(" ", strip=True) if abstract else ""
    })

/var/folders/73/3gm86nbd1vvgb4zm6ybrzx440000gn/T/ipykernel_4468/713189966.py:28: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(papers_xml, "lxml")


In [10]:
#2. Load models (scispaCy + BioNLP13CG)

# Base scientific model
nlp_sci = spacy.load("en_core_sci_sm")

# BioNLP model (biomedical entities)
nlp_bionlp = spacy.load("en_ner_bionlp13cg_md")



In [13]:
#4. Extract biomedical entities (BioNLP13CG + scispaCy) paper by paper
import pandas as pd

results = []

for i, paper in enumerate(papers_list):

    text = paper["abstract"]

    if not text:
        continue

    # safe limit
    text = text[:5000]

    doc_sci = nlp_sci(text)
    doc_bio = nlp_bio(text)

    # scispaCy entities
    for ent in doc_sci.ents:
        results.append({
            "paper_id": i,
            "title": paper["title"],
            "text": ent.text,
            "label": ent.label_,
            "model": "scispaCy"
        })

    # BioNLP entities
    for ent in doc_bio.ents:
        results.append({
            "paper_id": i,
            "title": paper["title"],
            "text": ent.text,
            "label": ent.label_,
            "model": "BioNLP13CG"
        })

In [11]:
#5: Convert to table
df = pd.DataFrame(results)

print(df.head())

df.to_csv("results/paper_by_paper_entities.csv", index=False)

   paper_id                                              title  \
0         0  The structure and function of taste G protein-...   
1         0  The structure and function of taste G protein-...   
2         0  The structure and function of taste G protein-...   
3         0  The structure and function of taste G protein-...   
4         0  The structure and function of taste G protein-...   

                                text   label     model  
0  Taste G protein-coupled receptors  ENTITY  scispaCy  
1                              GPCRs  ENTITY  scispaCy  
2                              sweet  ENTITY  scispaCy  
3                             TAS1Rs  ENTITY  scispaCy  
4    bitter (TAS2Rs) taste receptors  ENTITY  scispaCy  


In [12]:
#6. Normalize variants

def normalize_variant(v):
    v = v.strip()

    # Protein normalization
    if v.startswith("p."):
        v = v.replace(" ", "")

    # DNA normalization
    if v.startswith("c."):
        v = v.upper()

    # SNP normalization
    if v.startswith("rs"):
        v = v.lower()

    return v

normalized_variants = [normalize_variant(v) for v in variants]

print("Normalized Variants:", normalized_variants)

NameError: name 'variants' is not defined

NameError: name 'genes' is not defined